<a href="https://colab.research.google.com/github/wevertonmonteiro/teste-tecnico-engenharia-de-dados/blob/main/Teste_engenheiro_de_software.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Desafio Tecnico Engenharia de Dados**

In [2]:
pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=e6956a61214421505c420cda213a766454234a612f4f6a4e15ce296e8a8fa282
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [3]:
# Importando as bibliotecas para utilizar na analise
import pandas as pd
from pandasql import sqldf

In [4]:
# Utilizo essa ferramente para executar consultas SQL nos objetos do Panda

pysqldf = lambda q: sqldf(q, globals())

In [6]:
# Carrego todos os arquivos que utilizarei na analise

buyers = pd.read_csv('/content/buyers.csv', sep=',')
order_items = pd.read_csv('/content/order_items.csv', sep=',')
orders = pd.read_csv('/content/orders.csv', sep=',')
payments = pd.read_csv('/content/payments.csv', sep=',')
products = pd.read_csv('/content/products.csv', sep=',')
sellers = pd.read_csv('/content/sellers.csv', sep=',')

In [79]:
# display(buyers.head())
# display(order_items.head())
display(orders.head())
# display(payments.head())
# display(products.head())
# display(sellers.head())


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


Desafio 1


Escreva uma query que retorne o faturamento bruto mensal dos últimos 12 meses,
considerando apenas pedidos com status 'completed' ou 'delivered'. Inclua também a
quantidade de pedidos e o ticket médio de cada mês. Ordene do mês mais recente ao mais
antigo.

In [156]:
# Partes isoladas do DESAFIO 1
# Soma total do faturamento dos status completed' ou 'delivered’
# Somar Faturamento bruto mensal
# Somar Quantidade de pedidos
# Somar Ticket médio
# Filtrar pelos ultimos 12 meses
# Incluir quantidade de pedidos e ticket medio de cada mes
# Ordernar do mais recente ao mais antigo

PandaSQLException: (sqlite3.OperationalError) near "WHERE": syntax error
[SQL: 
SELECT SUM("total_value") AS faturamento_total_bruto from orders,  
WHERE "status" = "completed" OR "delivered"
LIMIT 10
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [203]:
# Utilizo o ('%Y-%m', "created_at") para agrupar por mes
# Desafio 1
query = """
SELECT
STRFTIME('%Y-%m', "created_at") AS "mês",
SUM(total_value) AS faturamento_total_bruto,
COUNT(id) AS quantidade_pedidos,
AVG(total_value) AS ticket_medio

FROM orders

WHERE (status = 'completed' OR status = 'delivered') AND
STRFTIME('%Y-%m', "created_at") BETWEEN '2023-12' AND '2024-11'

GROUP BY STRFTIME('%Y-%m', "created_at")
ORDER BY "mês" DESC
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


,mês,faturamento_total_bruto,quantidade_pedidos,ticket_medio
0,2024-11,56710238.63,3643,15566.906020
1,2024-10,62493439.13,3863,16177.437000
2,2024-09,60274462.50,3779,15949.844536
3,2024-08,60102539.82,3743,16057.317612
4,2024-07,59769406.95,3862,15476.283519
5,2024-06,58797774.37,3644,16135.503395
6,2024-05,60730315.72,3848,15782.306580
7,2024-04,57779312.59,3653,15816.948423
8,2024-03,61525913.64,3877,15869.464442
9,2024-02,57926376.26,3628,15966.476367


Desafio 2
A diretora comercial quer um ranking dos 10 sellers com maior crescimento de GMV entre o
trimestre atual e o anterior, mas só quer ver sellers que tiveram pelo menos 50 pedidos em ambos
os trimestres (para evitar distorção de sellers novos ou inativos).
Escreva a query que resolve esse problema, exibindo: nome do seller, estado, GMV do
trimestre anterior, GMV do trimestre atual e o percentual de crescimento. Ordene pelo maior
crescimento.

In [ ]:
# Desafio 2
query = """
SELECT *
FROM buyers
LIMIT 3
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

Desafio 3
O time de fraudes suspeita que alguns sellers estão aplicando descontos abusivos para inflar
volume artificialmente. Você precisa encontrar todos os pedidos onde o desconto total (soma dos
descontos dos itens) representa mais de 40% do valor bruto do pedido, listando também o seller
responsável e a data do pedido. Exclua pedidos cancelados.
Escreva a query e explique brevemente o raciocínio por trás dela.

In [39]:
# Desafio 3
query = """
SELECT *
FROM buyers
LIMIT 3
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01


Desafio 4
Existe um produto com comportamento estranho: ele tem um volume de vendas alto, mas nunca
aparece como item mais vendido dentro de nenhum pedido (nunca é o item de maior valor num
pedido). Encontre todos os produtos que se encaixam nessa descrição: total de unidades vendidas
> 1.000, mas que em nenhum pedido foram o item de maior valor unitário.
Escreva a query.
Dica: window functions podem ser suas aliadas aqui e avalie os resultados e viabilidade da
análise e faça os questionamentos necessários.

In [ ]:
# Desafio 4
query = """
SELECT *
FROM buyers
LIMIT 3
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4